In [ ]:
"""
Module: SOLO Doublet Detection and Exclusion for Sample Mock1
Description: Employs the deep generative scVI and SOLO framework to detect, 
             visualize, and exclude doublet cells from the 10x Genomics raw dataset.
"""

import os
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import seaborn as sns
import matplotlib.pyplot as plt

# ==========================================
# 1. Environmental & Path Configurations
# ==========================================
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=80, facecolor="white")
print(sc.logging.print_header())

# Masked paths configured via environment or relative mappings for secure deployment
DATA_DIR = Path(os.getenv("SC_DATA_DIR", "./data/raw/Zikv1/filtered_feature_bc_matrix"))

# ==========================================
# 2. Raw Data Ingestion (Mock1 Sample)
# ==========================================
adata = sc.read_10x_mtx(DATA_DIR, var_names="gene_symbols", cache=True)
adata.var_names_make_unique()

print(f"Initial Cell Number: {adata.n_obs}")
print(f"Initial Gene Number: {adata.n_vars}")

# ==========================================
# 3. Feature Selection & Model Training
# ==========================================
# Feature selection optimizing training speed and biological variation capture
sc.pp.filter_genes(adata, min_cells=10)
sc.pp.highly_variable_genes(adata, n_top_genes=2000, subset=True, flavor='seurat_v3')

# Train deep generative scVI base model
scvi.model.SCVI.setup_anndata(adata)
vae = scvi.model.SCVI(adata)
vae.train()

# Initialize and fit SOLO classifier using latent representations
solo = scvi.external.SOLO.from_scvi_model(vae)
solo.train()

# ==========================================
# 4. Doublet Prediction & Post-filtering
# ==========================================
df = solo.predict()
df['prediction'] = solo.predict(soft=False)
df.index = df.index.map(lambda x: x[:-2])  # Trim barcode suffix adjustments
df['dif'] = df.doublet - df.singlet

print("Doublet Classification Statistics:")
print(df.groupby('prediction').count())

# Filter high-confidence doublets (Score Difference > 1)
doublets = df[(df.prediction == 'doublet') & (df.dif > 1)]

# Re-load complete raw matrix and purge identified doublets
adata_final = sc.read_10x_mtx(DATA_DIR, var_names="gene_symbols", cache=True)
adata_final.var_names_make_unique()

# Append doublet labels to metadata and filter
adata_final.obs['doublet'] = adata_final.obs.index.isin(doublets.index)
adata_cleaned = adata_final[~adata_final.obs.doublet].copy()

print(f"Purged {len(doublets)} doublets from the dataset.")
print(f"Cleaned Cell Number: {adata_cleaned.n_obs}")